# 02 · Train and compare Poisson GAM profiles

**The training cell below trains real models.** It learns one smooth curve for
each detector and flight phase, then saves the fitted models and predictions.
Run cells in order with the **Python (HERA)** kernel.

These models describe the selected flight. Errors measured on the fitting data
do not demonstrate accuracy on a new flight. The displayed mean-curve intervals
exclude calibration/systematic error and are not peak-location intervals.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image

# Works when Jupyter starts in the repository or its notebooks folder.
here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents)
             if (p / "src/prepare_profiles.py").is_file()
             and (p / "src/fit_profile.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside HERA-Machine-Learning/notebooks.")
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT)
print("Python:", sys.executable)

## Choose the experiment

Keep these baseline settings first. `elapsed` assumes continuous acquisition.
`candidate-live` additionally assumes the recorded deadtime is cumulative seconds.
Choose one event channel per experiment; coincidence events are a subset of all
events, so the two channels are not independent datasets.

In [ ]:
FLIGHT_ID = "2025-06-01"
DETECTORS = ["top", "bottom"]
BIN_SECONDS = 10
CHANNEL = "all"                 # "all" or "coincident"
EXPOSURE = "elapsed"            # "elapsed" or "candidate-live"
N_SPLINES = 12
FIXED_LAMBDA = None              # None selects smoothing by in-sample UBRE

# Optional: pin a prepared CSV for an exact rerun. Paths may be relative to ROOT.
# Example: {"top": "clean/2025-06-01/RUN_ID/top_bins_10s.csv"}
INPUT_OVERRIDES = {}

## Select and inspect the training inputs

Automatic selection uses the newest matching preparation directory, not the
newest file from an arbitrary detector. Confirm the printed paths and selections.
No training occurs in this cell.

In [ ]:
import math
import numpy as np
import fit_profile

if CHANNEL not in ("all", "coincident") or EXPOSURE not in fit_profile.EXPOSURES:
    raise ValueError("Check CHANNEL and EXPOSURE.")
if not math.isfinite(BIN_SECONDS) or BIN_SECONDS <= 0:
    raise ValueError("BIN_SECONDS must be positive and finite.")
if not DETECTORS or len(DETECTORS) != len(set(DETECTORS)):
    raise ValueError("Choose at least one detector, without duplicates.")

inputs, preprocessing, selections = {}, {}, []
for detector in DETECTORS:
    if detector in INPUT_OVERRIDES:
        chosen = ROOT / Path(INPUT_OVERRIDES[detector])
    else:
        pattern = f"*/{detector}_bins_{float(BIN_SECONDS):g}s.csv"
        candidates = sorted((ROOT / "clean" / FLIGHT_ID).glob(pattern))
        if not candidates:
            raise FileNotFoundError(f"Run notebook 01 for {detector}, {BIN_SECONDS}s bins first.")
        chosen = candidates[-1]
    chosen = chosen.resolve()
    checked = fit_profile.load_bins(chosen, CHANNEL, EXPOSURE)
    if not (checked.flight_id.eq(FLIGHT_ID).all() and checked.detector_id.eq(detector).all()
            and np.allclose(checked.elapsed_s, BIN_SECONDS)):
        raise ValueError(f"Selected CSV does not match the experiment: {chosen}")
    report_path = ROOT / "results" / FLIGHT_ID / chosen.parent.name / "preparation_report.json"
    report = json.loads(report_path.read_text(encoding="utf-8"))
    inputs[detector] = chosen
    preprocessing[detector] = {"source_sha256": report["source_sha256"],
        "configuration_except_bin_width": {k: v for k, v in report["config"].items()
                                           if k != "bin_seconds"}}
    print(detector.upper(), "input:", chosen)
    counts = checked.fit_selection.value_counts().to_dict()
    selections.append({"detector": detector, **counts})
display(pd.DataFrame(selections).fillna(0))

## Train the models

**This cell performs training.** It calls `src/fit_profile.py` with integer event
counts and explicit exposure. Ascent and descent are fitted separately. The
fitter selects smoothing from a small grid unless `FIXED_LAMBDA` is supplied.

Every run writes a new results directory, including `gam_models.pkl`. Existing
inputs, their verification flags and earlier model runs remain unchanged.

In [ ]:
trained_runs = {}
for detector, csv_path in inputs.items():
    arguments = ["--input", str(csv_path), "--channel", CHANNEL,
                 "--exposure", EXPOSURE, "--n-splines", str(N_SPLINES),
                 "--output-root", str(ROOT / "results")]
    if FIXED_LAMBDA is not None:
        arguments.extend(["--lam", str(FIXED_LAMBDA)])
    print("TRAINING:", detector)
    trained_runs[detector] = fit_profile.main(arguments)

## Review the fitted profiles and residuals

Points are measured bin rates; the curve is learned from those points. Check
whether residuals show a systematic pattern. A maximum at the observed boundary
does not locate the physical peak. No calibrated altitude is inferred here.

In [ ]:
reports, curves = {}, {}
for detector, folder in trained_runs.items():
    reports[detector] = json.loads((folder / "fit_summary.json").read_text(encoding="utf-8"))
    curves[detector] = pd.read_csv(folder / "profile_curves.csv")
    print(detector.upper(), "model outputs:", folder)
    display(Image(filename=str(folder / "gam_profile.png"), width=1000))

In [ ]:
rows = []
for detector, report in reports.items():
    for phase, stats in report["phases"].items():
        peak = stats["fitted_maximum"]
        rows.append({"detector": detector, "phase": phase,
                     "bins": stats["bins"], "events": stats["events"],
                     "candidate_peak_depth_g_cm2": peak["depth_g_cm2"],
                     "candidate_peak_rate_cps": peak["rate_cps"],
                     "peak_status": peak["status"],
                     "in_sample_mae_cps": stats["in_sample_rate_mae_cps"],
                     "pearson_chi2_per_df": stats["pearson_chi2_per_residual_df"],
                     "lambda": stats["selected_lambda"]})
comparison = pd.DataFrame(rows)
display(comparison.round(3))
print("These are fitting-data diagnostics, not held-out prediction scores.")

## Compare detectors and save the experiment

Each curve uses its detector's own pressure. Differences may reflect detector
response or pressure measurement as well as the radiation field. These curves
have not been calibrated to a common detector response.

In [ ]:
from datetime import datetime, timezone
import matplotlib.pyplot as plt

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
comparison_dir = ROOT / "results" / FLIGHT_ID / f"{stamp}_notebook_compare"
comparison_dir.mkdir(parents=True, exist_ok=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for number, (detector, curve) in enumerate(curves.items()):
    color = plt.get_cmap("tab10")(number % 10)
    for ax, phase in zip(axes, ("ascent", "descent")):
        part = curve[curve.phase_provisional.eq(phase)]
        ax.plot(part.depth_g_cm2, part.fitted_rate_cps, label=detector, color=color)
        ax.fill_between(part.depth_g_cm2, part.mean_rate_lower95_conditional,
                        part.mean_rate_upper95_conditional, alpha=0.12, color=color)
        ax.set(xscale="log", title=phase.capitalize(),
               xlabel="Atmospheric depth [g/cm²; log scale]")
        ax.grid(alpha=0.2)
axes[0].set_ylabel("Recorded events / exposure second")
axes[1].legend()
fig.suptitle(f"Exploratory GAMs: {FLIGHT_ID}, {BIN_SECONDS}s bins, {CHANNEL}, {EXPOSURE}")
fig.text(0.5, 0.025, "Shading: 95% conditional mean intervals; calibration/systematic error excluded.\n"
         "Each detector uses its own pressure. Responses are not cross-calibrated.",
         ha="center", fontsize=9)
fig.tight_layout(rect=(0, 0.10, 1, 0.94))
plot_path = comparison_dir / "detector_comparison.png"
fig.savefig(plot_path, dpi=160)
plt.close(fig)
display(Image(filename=str(plot_path), width=1000))

comparison.to_csv(comparison_dir / "fit_comparison.csv", index=False)
# The history table compares only runs with these same non-binning choices.
comparison_key = {"flight": FLIGHT_ID, "channel": CHANNEL, "exposure": EXPOSURE,
                  "n_splines": N_SPLINES, "fixed_lambda": FIXED_LAMBDA,
                  "fitter_sha256": fit_profile.fingerprint(fit_profile.__file__),
                  "preprocessing": preprocessing,
                  "package_versions": reports[DETECTORS[0]]["versions"]}
manifest = {"comparison_key": comparison_key, "bin_seconds": float(BIN_SECONDS),
            "inputs": {k: str(v) for k, v in inputs.items()},
            "model_output_directories": {k: str(v) for k, v in trained_runs.items()}}
(comparison_dir / "comparison_run.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Comparison saved:", comparison_dir)

## Compare bin-size experiments

This table shows your latest matching run at each bin width. It keeps flight,
channel, exposure, model settings, source recordings and non-binning preparation
choices consistent. It will initially contain only the 10-second baseline.

In [ ]:
history_frames = []
for manifest_path in sorted((ROOT / "results" / FLIGHT_ID).glob("*_notebook_compare/comparison_run.json")):
    previous = json.loads(manifest_path.read_text(encoding="utf-8"))
    if previous["comparison_key"] != comparison_key:
        continue
    frame = pd.read_csv(manifest_path.parent / "fit_comparison.csv")
    frame["bin_seconds"] = previous["bin_seconds"]
    frame["run"] = manifest_path.parent.name
    history_frames.append(frame)
history = pd.concat(history_frames, ignore_index=True).sort_values("run")
history = history.drop_duplicates(["detector", "phase", "bin_seconds"], keep="last")
history = history.sort_values(["detector", "phase", "bin_seconds"])
display(history[["detector", "phase", "bin_seconds", "bins",
                 "candidate_peak_depth_g_cm2", "candidate_peak_rate_cps",
                 "peak_status", "in_sample_mae_cps"]].round(3))
history.to_csv(comparison_dir / "bin_width_history.csv", index=False)

## What comes next

1. Keep this 10-second baseline. In notebook 01, change only `BIN_SECONDS` to
   **20**, then rerun it. Set 20 here and rerun this notebook. Repeat with **30**.
   The history table above will show whether the candidate peak moves.
2. Review peak stability, coverage and residual patterns. Wider bins naturally
   reduce count noise; the lowest fitting-data MAE does not establish the best
   bin width. These sensitivity runs are not independent validation datasets.
3. Resolve pressure/deadtime details and identify calibration records for these
   detectors. Keep resulting changes documented before final peak claims.
4. Prepare another suitable flight using its own configuration. For claims about
   predicting new flights, keep entire flights out of training and tuning.

Before committing notebooks, use **Edit → Clear Outputs of All Cells** and save.
Commit notebooks, scripts, configurations and dependency files. Keep raw data,
generated tables, plots, models and `.venv` outside Git.